In [1]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")

In [2]:
from helper import create_tokenized_word_list
tokenized = create_tokenized_word_list(dataset)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tahas44/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
from helper import create_tokenized_word_list_for_query
query_tokenized = create_tokenized_word_list_for_query(dataset)
print("Queries tokenized!")

Queries tokenized!


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tahas44/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
query_text_based = []

for query_list in query_tokenized:
    query_text_based.append(" ".join(query_list))

In [4]:
from nltk.stem import PorterStemmer
porter_stemmer = PorterStemmer()
stemmed_texts = []

for word_list in tokenized:
    stemmed_words = [porter_stemmer.stem(word) for word in word_list]
    stemmed_texts.append(stemmed_words)

print("texts stemmed!")

texts stemmed!


In [5]:
stemmed_texts

[['use',
  'land',
  'craft',
  'world',
  'war',
  'ii',
  'use',
  'today',
  'privat',
  'boat',
  'train',
  'facil',
  '6',
  '71',
  'inlin',
  'six',
  'cylind',
  'diesel',
  'engin',
  '71',
  'refer',
  'displac',
  'cubic',
  'inch',
  'cylind',
  'fire',
  'order',
  'engin',
  '1',
  '5',
  '3',
  '6',
  '2',
  '4',
  'engin',
  'compress',
  'ratio',
  '18',
  '7',
  '1',
  '4',
  '250',
  'inch',
  'bore',
  '5',
  '00',
  'inch',
  'stroke',
  'engin',
  'weigh',
  '54',
  'inch',
  'long',
  '29',
  'inch',
  'wide',
  '41',
  'inch',
  'tall',
  '2',
  '100',
  'revolut',
  'per',
  'minut',
  'engin',
  'capabl',
  'produc',
  '230',
  'hors',
  'power',
  '172',
  'kilowatt',
  'v',
  'type',
  'version',
  '71',
  'seri',
  'develop',
  '1957',
  '6',
  '71',
  'two',
  'stroke',
  'engin',
  'engin',
  'natur',
  'aspir',
  'air',
  'provid',
  'via',
  'root',
  'type',
  'blower',
  'howev',
  '6',
  '71t',
  'model',
  'turbocharg',
  'supercharg',
  'util',
  

In [6]:
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi(stemmed_texts)

In [8]:
scores_of_all_queries = []

for query in query_tokenized:
    scores_of_all_queries.append(bm25.get_scores(query))

In [10]:
len(scores_of_all_queries)

1444

In [13]:
from collections import defaultdict
from helper import Scoredoc

doc_dict = defaultdict(str)

for i, doc in enumerate(dataset.docs_iter()):
    doc_dict[i] = doc.doc_id

doc_dict = dict(doc_dict)

qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

score_doc_dict = dict(score_doc_dict)

print("Necessary dicts created!")

Necessary dicts created!


In [12]:
import pandas as pd
query_ids = [query.query_id for query in dataset.queries_iter()]
df = pd.DataFrame(query_ids, columns=["Query_ID"])
df

,Query_ID
0,123839
1,188629
2,13898
3,316959
4,515031
...,...
1439,896124
1440,12319
1441,4421
1442,296526


In [13]:
from helper import create_AP, create_ndcg, create_statistical_columns
df = create_statistical_columns(df, qrels_dict, doc_dict, scores_of_all_queries)
df = create_AP(df, qrels_dict, doc_dict, scores_of_all_queries)
df = create_ndcg(df, doc_dict, scores_of_all_queries, score_doc_dict)

In [14]:
df

,Query_ID,recall_5,recall_10,precision_5,precision_10,f_score_5,f_score_10,AP_5,AP_10,NDCG_5,NDCG_10
0,123839,50.000000,100.000000,60.0,60.0,54.545455,75.000000,0.458333,0.768849,0.883923,0.852981
1,188629,33.333333,33.333333,40.0,20.0,36.363636,25.000000,0.333333,0.333333,0.889669,0.918338
2,13898,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.188244,0.238403
3,316959,11.111111,22.222222,20.0,20.0,14.285714,21.052632,0.111111,0.148148,1.000000,0.998975
4,515031,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.892079,0.924320
...,...,...,...,...,...,...,...,...,...,...,...
1439,896124,12.500000,12.500000,20.0,10.0,15.384615,11.111111,0.125000,0.125000,0.464253,0.502333
1440,12319,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.197367,0.189544
1441,4421,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.495470,0.468331
1442,296526,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.093400


In [15]:
mydict = {
    "Method": "BM25 - Porter's Stemmer",
    "recall_5_mean": df["recall_5"].mean(),
    "recall_5_std": df["recall_5"].std(),
    "recall_5_max": df["recall_5"].max(),
    "recall_5_min": df["recall_5"].min(),
    "recall_10_mean": df["recall_10"].mean(),
    "recall_10_std": df["recall_10"].std(),
    "recall_10_max": df["recall_10"].max(),
    "recall_10_min": df["recall_10"].min(),
    "precision_5_mean": df["precision_5"].mean(),
    "precision_5_std": df["precision_5"].std(),
    "precision_5_max": df["precision_5"].max(),
    "precision_5_min": df["precision_5"].min(),
    "precision_10_mean": df["precision_10"].mean(),
    "precision_10_std": df["precision_10"].std(),
    "precision_10_max": df["precision_10"].max(),
    "precision_10_min": df["precision_10"].min(),
    "f_score_5_mean": df["f_score_5"].mean(),
    "f_score_5_std": df["f_score_5"].std(),
    "f_score_5_max": df["f_score_5"].max(),
    "f_score_5_min": df["f_score_5"].min(),
    "f_score_10_mean": df["f_score_10"].mean(),
    "f_score_10_std": df["f_score_10"].std(),
    "f_score_10_max": df["f_score_10"].max(),
    "f_score_10_min": df["f_score_10"].min(),
    "MAP_5": df["AP_5"].mean(),
    "MAP_10": df["AP_10"].mean(),
    "NDCG_5_mean": df["NDCG_5"].mean(),
    "NDCG_5_std": df["NDCG_5"].std(),
    "NDCG_5_max": df["NDCG_5"].max(),
    "NDCG_5_min": df["NDCG_5"].min(),
    "NDCG_10_mean": df["NDCG_10"].mean(),
    "NDCG_10_std": df["NDCG_10"].std(),
    "NDCG_10_max": df["NDCG_10"].max(),
    "NDCG_10_min": df["NDCG_10"].min()
}

In [16]:
df_csv = pd.DataFrame(mydict, index=[0])
df_csv

,Method,recall_5_mean,recall_5_std,recall_5_max,recall_5_min,recall_10_mean,recall_10_std,recall_10_max,recall_10_min,precision_5_mean,...,MAP_5,MAP_10,NDCG_5_mean,NDCG_5_std,NDCG_5_max,NDCG_5_min,NDCG_10_mean,NDCG_10_std,NDCG_10_max,NDCG_10_min
0,BM25 - Porter's Stemmer,9.35294,13.872295,83.333333,0.0,13.876176,19.458257,100.0,0.0,19.044321,...,0.072112,0.091605,0.541213,0.369604,1.0,0.0,0.548739,0.365356,1.0,0.0


In [17]:
df_csv.to_parquet("BM25PorterStemmer.parquet")
print("Parquet dosyası kaydedildi!")

Parquet dosyası kaydedildi!


### Stem the Query vectors

In [7]:
stemmed_queries = []

for query_list in query_tokenized:
    stemmed_query_words = [porter_stemmer.stem(word) for word in query_list]
    stemmed_queries.append(stemmed_query_words)

print("Queries stemmed!")

Queries stemmed!


In [8]:
stemmed_queries

[['yanni'],
 ['k', 'pop'],
 ['venic', 'film', 'festiv'],
 ['downtown', 'brooklyn'],
 ['pennsylvania', 'hous', 'repres'],
 ['northern', 'premier', 'leagu'],
 ['first', 'nation', 'pictur'],
 ['poland'],
 ['cyperacea'],
 ['trinidad', 'tobago'],
 ['normandi', 'land'],
 ['perri', 'index'],
 ['arm', 'architectur'],
 ['mi5'],
 ['patriarch', 'antioch'],
 ['nashik', 'district'],
 ['semiot'],
 ['santa', 'cruz', 'de', 'tenerif'],
 ['gastropoda'],
 ['zaragoza'],
 ['south', 'carolina'],
 ['augustin', 'hippo'],
 ['canton', 'geneva'],
 ['leo', 'tolstoy'],
 ['riemannian', 'manifold'],
 ['tuzla'],
 ['subaru'],
 ['tomsk'],
 ['unit', 'state', 'depart', 'agricultur'],
 ['north', 'borneo'],
 ['isaac', 'asimov'],
 ['hilbert', 'space'],
 ['chechen'],
 ['arunach', 'pradesh'],
 ['central', 'otago'],
 ['ira', 'gershwin'],
 ['rock', 'roll'],
 ['fiba'],
 ['east', 'coast', 'unit', 'state'],
 ['wikipedia'],
 ['pernambuco'],
 ['west', 'coast', 'eagl'],
 ['twin', 'peak'],
 ['dutch', 'languag'],
 ['liber', 'parti', 'c

In [9]:
query_tokenized

[['yanni'],
 ['k', 'pop'],
 ['venice', 'film', 'festival'],
 ['downtown', 'brooklyn'],
 ['pennsylvania', 'house', 'representatives'],
 ['northern', 'premier', 'league'],
 ['first', 'national', 'pictures'],
 ['poland'],
 ['cyperaceae'],
 ['trinidad', 'tobago'],
 ['normandy', 'landings'],
 ['perry', 'index'],
 ['arm', 'architecture'],
 ['mi5'],
 ['patriarch', 'antioch'],
 ['nashik', 'district'],
 ['semiotics'],
 ['santa', 'cruz', 'de', 'tenerife'],
 ['gastropoda'],
 ['zaragoza'],
 ['south', 'carolina'],
 ['augustine', 'hippo'],
 ['canton', 'geneva'],
 ['leo', 'tolstoy'],
 ['riemannian', 'manifold'],
 ['tuzla'],
 ['subaru'],
 ['tomsk'],
 ['united', 'states', 'department', 'agriculture'],
 ['north', 'borneo'],
 ['isaac', 'asimov'],
 ['hilbert', 'space'],
 ['chechens'],
 ['arunachal', 'pradesh'],
 ['central', 'otago'],
 ['ira', 'gershwin'],
 ['rock', 'roll'],
 ['fiba'],
 ['east', 'coast', 'united', 'states'],
 ['wikipedia'],
 ['pernambuco'],
 ['west', 'coast', 'eagles'],
 ['twin', 'peaks'],

In [25]:
len(stemmed_queries)

1444

In [10]:
scores_of_all_queries_queries_stemmed = []

for query in stemmed_queries:
    scores_of_all_queries_queries_stemmed.append(bm25.get_scores(query))

In [11]:
scores_of_all_queries_queries_stemmed

[array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0.        , 2.32798045, 0.        , ..., 0.        , 0.        ,
        0.        ], shape=(369721,)),
 array([ 0.        ,  0.        , 10.41618338, ...,  0.        ,
         0.        ,  0.        ], shape=(369721,)),
 array([0.        , 0.        , 0.        , ..., 0.43415237, 1.6423582 ,
        0.        ], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([2.51402525, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shap

In [12]:
len(scores_of_all_queries_queries_stemmed)

1444

In [14]:
import pandas as pd
query_ids = [query.query_id for query in dataset.queries_iter()]
df_new = pd.DataFrame(query_ids, columns=["Query_ID"])
df_new

,Query_ID
0,123839
1,188629
2,13898
3,316959
4,515031
...,...
1439,896124
1440,12319
1441,4421
1442,296526


In [15]:
from helper import create_AP, create_ndcg, create_statistical_columns
df_new = create_statistical_columns(df_new, qrels_dict, doc_dict, scores_of_all_queries_queries_stemmed)
df_new = create_AP(df_new, qrels_dict, doc_dict, scores_of_all_queries_queries_stemmed)
df_new = create_ndcg(df_new, doc_dict, scores_of_all_queries_queries_stemmed, score_doc_dict)

In [16]:
df_new

,Query_ID,recall_5,recall_10,precision_5,precision_10,f_score_5,f_score_10,AP_5,AP_10,NDCG_5,NDCG_10
0,123839,50.000000,100.000000,60.0,60.0,54.545455,75.000000,0.458333,0.768849,0.883923,0.852981
1,188629,33.333333,33.333333,40.0,20.0,36.363636,25.000000,0.333333,0.333333,0.889669,0.918338
2,13898,33.333333,33.333333,40.0,20.0,36.363636,25.000000,0.333333,0.333333,0.000000,0.000000
3,316959,11.111111,22.222222,20.0,20.0,14.285714,21.052632,0.111111,0.148148,1.000000,0.998975
4,515031,0.000000,14.285714,0.0,20.0,0.000000,16.666667,0.000000,0.027778,0.551929,0.471942
...,...,...,...,...,...,...,...,...,...,...,...
1439,896124,12.500000,12.500000,20.0,10.0,15.384615,11.111111,0.125000,0.125000,0.464253,0.502333
1440,12319,4.545455,4.545455,20.0,10.0,7.407407,6.250000,0.045455,0.045455,0.197103,0.200593
1441,4421,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.495470,0.468331
1442,296526,10.000000,10.000000,20.0,10.0,13.333333,10.000000,0.050000,0.050000,0.535444,0.561980


In [18]:
mydict = {
    "Method": "BM25 - Porter's Stemmer Queries Stemmed",
    "recall_5_mean": df_new["recall_5"].mean(),
    "recall_5_std": df_new["recall_5"].std(),
    "recall_5_max": df_new["recall_5"].max(),
    "recall_5_min": df_new["recall_5"].min(),
    "recall_10_mean": df_new["recall_10"].mean(),
    "recall_10_std": df_new["recall_10"].std(),
    "recall_10_max": df_new["recall_10"].max(),
    "recall_10_min": df_new["recall_10"].min(),
    "precision_5_mean": df_new["precision_5"].mean(),
    "precision_5_std": df_new["precision_5"].std(),
    "precision_5_max": df_new["precision_5"].max(),
    "precision_5_min": df_new["precision_5"].min(),
    "precision_10_mean": df_new["precision_10"].mean(),
    "precision_10_std": df_new["precision_10"].std(),
    "precision_10_max": df_new["precision_10"].max(),
    "precision_10_min": df_new["precision_10"].min(),
    "f_score_5_mean": df_new["f_score_5"].mean(),
    "f_score_5_std": df_new["f_score_5"].std(),
    "f_score_5_max": df_new["f_score_5"].max(),
    "f_score_5_min": df_new["f_score_5"].min(),
    "f_score_10_mean": df_new["f_score_10"].mean(),
    "f_score_10_std": df_new["f_score_10"].std(),
    "f_score_10_max": df_new["f_score_10"].max(),
    "f_score_10_min": df_new["f_score_10"].min(),
    "MAP_5": df_new["AP_5"].mean(),
    "MAP_10": df_new["AP_10"].mean(),
    "NDCG_5_mean": df_new["NDCG_5"].mean(),
    "NDCG_5_std": df_new["NDCG_5"].std(),
    "NDCG_5_max": df_new["NDCG_5"].max(),
    "NDCG_5_min": df_new["NDCG_5"].min(),
    "NDCG_10_mean": df_new["NDCG_10"].mean(),
    "NDCG_10_std": df_new["NDCG_10"].std(),
    "NDCG_10_max": df_new["NDCG_10"].max(),
    "NDCG_10_min": df_new["NDCG_10"].min()
}

In [19]:
df_parquet = pd.DataFrame(mydict, index=[0])
df_parquet

,Method,recall_5_mean,recall_5_std,recall_5_max,recall_5_min,recall_10_mean,recall_10_std,recall_10_max,recall_10_min,precision_5_mean,...,MAP_5,MAP_10,NDCG_5_mean,NDCG_5_std,NDCG_5_max,NDCG_5_min,NDCG_10_mean,NDCG_10_std,NDCG_10_max,NDCG_10_min
0,BM25 - Porter's Stemmer Queries Stemmed,14.478316,14.481887,83.333333,0.0,20.702477,19.543478,100.0,0.0,29.66759,...,0.114719,0.14149,0.50807,0.349112,1.0,0.0,0.50813,0.345892,1.0,0.0


In [20]:
df_parquet.to_parquet("BM25PorterStemmerQueriesStemmed.parquet")
print("Parquet!")

Parquet!
